In [1]:
import os
import pandas as pd
import re

# ==============================
# CONFIGURATION
# ==============================

RAW_DATA_PATH = "data/raw"
MAPPING_PATH = "mappings"
OUTPUT_PATH = "output"

TARGET_QUARTER = "Q4"

PROVIDERS = [
    "Provider_A.xlsx",
    "Provider_B.xlsx",
    "Provider_C.xlsx",
    "Provider_D.xlsx",
    "Provider_E.xlsx",
]

# ==============================
# HELPER FUNCTIONS
# ==============================

def get_column(df, possible_names):
    """Find column dynamically"""
    df_cols = [col.lower().strip() for col in df.columns]

    for name in possible_names:
        if name.lower() in df_cols:
            return df.columns[df_cols.index(name.lower())]

    return None


def clean_market_value(value):
    if pd.isna(value):
        return None
    value = str(value)
    value = re.sub(r"[₹$,]", "", value)
    try:
        return float(value)
    except:
        return None


def add_metadata(df, provider, file_name):
    df["Provider"] = provider
    df["Source File"] = file_name
    return df


# ==============================
# COLUMN STANDARDIZATION
# ==============================

def standardize_columns(df, column_map, provider):
    column_map.columns = column_map.columns.str.strip()

    # Case 1: LONG FORMAT
    if any("provider" in col.lower() for col in column_map.columns):

        column_map.columns = column_map.columns.str.lower()

        provider_col = [c for c in column_map.columns if "provider" in c][0]
        source_col = [c for c in column_map.columns if "source" in c][0]
        target_col = [c for c in column_map.columns if "target" in c or "standard" in c][0]

        mapping = column_map[
            column_map[provider_col].astype(str).str.lower() == provider.lower()
        ]

        if mapping.empty:
            print(f"⚠️ No mapping found for {provider}")
            return df

        mapping_dict = dict(zip(mapping[source_col], mapping[target_col]))
        return df.rename(columns=mapping_dict)

    # Case 2: WIDE FORMAT
    else:
        if "Standard Column" in column_map.columns:
            standard_col = "Standard Column"
        elif "Target Column" in column_map.columns:
            standard_col = "Target Column"
        else:
            print("⚠️ No standard column found in mapping file")
            return df

        if provider not in column_map.columns:
            print(f"⚠️ Provider column '{provider}' not found in mapping file")
            return df

        mapping_dict = dict(zip(column_map[provider], column_map[standard_col]))
        return df.rename(columns=mapping_dict)


# ==============================
# LOADER
# ==============================

def read_all_sheets(file_path):
    try:
        sheets = pd.read_excel(file_path, sheet_name=None)
        df_list = []

        for sheet_name, df in sheets.items():
            if not df.empty:
                df_list.append(df)

        return pd.concat(df_list, ignore_index=True)

    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return pd.DataFrame()


# ==============================
# TRANSFORMATION
# ==============================

def transform_data(df, provider, file_name, column_map, fund_map, asset_map):

    # Standardize columns
    df = standardize_columns(df, column_map, provider)

    # Clean Market Value
    if "Market Value" in df.columns:
        df["Market Value"] = df["Market Value"].apply(clean_market_value)

    # ======================
    # FUND MAPPING
    # ======================
    fund_col = get_column(df, ["Fund Name", "Fund", "Scheme Name"])

    if fund_col:
        df = df.rename(columns={fund_col: "Fund Name"})

        fund_map.columns = fund_map.columns.str.strip()

        map_fund_col = get_column(fund_map, ["Fund Name"])
        std_fund_col = get_column(fund_map, ["Standard Fund Name", "Standard Name"])

        if map_fund_col and std_fund_col:
            fund_map = fund_map.rename(columns={
                map_fund_col: "Fund Name",
                std_fund_col: "Standard Fund Name"
            })

            df = df.merge(fund_map, how="left", on="Fund Name")
            df["Fund Name"] = df["Standard Fund Name"].fillna(df["Fund Name"])

        else:
            print("⚠️ Fund mapping columns missing")

    else:
        print(f"⚠️ Fund column not found in {provider}")

    # ======================
    # ASSET CLASS MAPPING
    # ======================
    isin_col = get_column(df, ["ISIN", "isin", "ISIN Code", "Security ID"])

    if isin_col:
        df = df.rename(columns={isin_col: "ISIN"})

        asset_map.columns = asset_map.columns.str.strip()
        asset_isin_col = get_column(asset_map, ["ISIN"])

        if asset_isin_col:
            asset_map = asset_map.rename(columns={asset_isin_col: "ISIN"})
            df = df.merge(asset_map, how="left", on="ISIN")
        else:
            print("⚠️ ISIN not found in asset mapping")

    else:
        print(f"⚠️ ISIN column not found in {provider}")

    # Add metadata
    df = add_metadata(df, provider, file_name)

    return df


# ==============================
# VALIDATION
# ==============================

def validate(df):
    print("\n🔍 VALIDATION REPORT")
    print("------------------------")
    print("Missing ISINs:", df["ISIN"].isna().sum())
    print("Duplicate Rows:", df.duplicated().sum())
    print("Null Market Values:", df["Market Value"].isna().sum())
    print("------------------------\n")


# ==============================
# MAIN PIPELINE
# ==============================

def run_pipeline():

    print("🚀 Starting Pipeline...\n")

    os.makedirs(OUTPUT_PATH, exist_ok=True)

    column_map = pd.read_excel(os.path.join(MAPPING_PATH, "column_mapping.xlsx"))
    fund_map = pd.read_excel(os.path.join(MAPPING_PATH, "fund_mapping.xlsx"))
    asset_map = pd.read_excel(os.path.join(MAPPING_PATH, "asset_class_mapping.xlsx"))

    final_data = []

    for file in PROVIDERS:

        file_path = os.path.join(RAW_DATA_PATH, file)
        provider_name = file.split(".")[0].strip()

        print(f"📂 Processing: {file}")

        df = read_all_sheets(file_path)

        if df.empty:
            print(f"⚠️ Skipping {file}")
            continue

        # Provider E special filter
        if provider_name == "Provider_E":
            quarter_col = get_column(df, ["Quarter", "Qtr"])
            if quarter_col:
                df = df[df[quarter_col].astype(str).str.contains(TARGET_QUARTER, na=False)]

        df = transform_data(df, provider_name, file, column_map, fund_map, asset_map)

        final_data.append(df)

    final_df = pd.concat(final_data, ignore_index=True)

    required_columns = [
        "Quarter", "Fund Name", "ISIN", "Security Name",
        "Asset Class", "Market Value", "Currency",
        "Provider", "Source File"
    ]

    for col in required_columns:
        if col not in final_df.columns:
            final_df[col] = None

    final_df = final_df[required_columns]

    validate(final_df)

    output_file = os.path.join(OUTPUT_PATH, "final_report_Q4.xlsx")
    final_df.to_excel(output_file, index=False)

    print(f"✅ Done! Output saved at: {output_file}")


# ==============================
# ENTRY POINT
# ==============================

if __name__ == "__main__":
    run_pipeline()

🚀 Starting Pipeline...

📂 Processing: Provider_A.xlsx
⚠️ Provider column 'Provider_A' not found in mapping file
⚠️ Fund mapping columns missing
⚠️ ISIN not found in asset mapping
📂 Processing: Provider_B.xlsx
⚠️ Provider column 'Provider_B' not found in mapping file
⚠️ Fund mapping columns missing
⚠️ ISIN not found in asset mapping
📂 Processing: Provider_C.xlsx
⚠️ Provider column 'Provider_C' not found in mapping file
⚠️ Fund mapping columns missing
⚠️ ISIN not found in asset mapping
📂 Processing: Provider_D.xlsx
⚠️ Provider column 'Provider_D' not found in mapping file
⚠️ Fund mapping columns missing
⚠️ ISIN not found in asset mapping
📂 Processing: Provider_E.xlsx
⚠️ Provider column 'Provider_E' not found in mapping file
⚠️ Fund mapping columns missing
⚠️ ISIN not found in asset mapping

🔍 VALIDATION REPORT
------------------------
Missing ISINs: 0
Duplicate Rows: 0
Null Market Values: 225
------------------------

✅ Done! Output saved at: output\final_report_Q4.xlsx


In [2]:
pd.read_excel(r"C:\Users\Admin\OneDrive\Desktop\ML_projects\Python_Automation\output\final_report_Q4.xlsx")

,Quarter,Fund Name,ISIN,Security Name,Asset Class,Market Value,Currency,Provider,Source File
0,NaN,Alpha Growth Fund,ISIN00000,Reliance,NaN,NaN,INR,Provider_A,Provider_A.xlsx
1,NaN,Beta Equity Fund,ISIN00001,US Treasury Bond,NaN,NaN,INR,Provider_A,Provider_A.xlsx
2,NaN,Alpha Growth Fund,ISIN00002,TCS,NaN,NaN,INR,Provider_A,Provider_A.xlsx
3,NaN,Gamma Income Fund,ISIN00003,US Treasury Bond,NaN,NaN,USD,Provider_A,Provider_A.xlsx
4,NaN,Beta Equity Fund,ISIN00004,US Treasury Bond,NaN,NaN,INR,Provider_A,Provider_A.xlsx
...,...,...,...,...,...,...,...,...,...
220,2025Q4,Gamma Income Fund,ISIN00010,Infosys,NaN,NaN,USD,Provider_E,Provider_E.xlsx
221,2025Q4,Beta Equity Fund,ISIN00011,Microsoft,NaN,NaN,INR,Provider_E,Provider_E.xlsx
222,2025Q4,Delta Opportunities,ISIN00012,Infosys,NaN,NaN,INR,Provider_E,Provider_E.xlsx
223,2025Q4,Gamma Income Fund,ISIN00013,TCS,NaN,NaN,USD,Provider_E,Provider_E.xlsx


In [ ]:
raw_data_path = "data/raw"
mapping_data_path = "mappings"
output_path = "output"

providers = [
    
]